In [1]:
#Reporteria automatizada
#importaciones
!pip install jinja2 weasyprint python-docx -q

print("Librerias instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 16.5 MB/s eta 0:00:00
Librerias instaladas correctamente


In [2]:
!git clone https://github.com/sebaas256/Proyecto_final_gestion_de_datos_IA.git

Cloning into 'Proyecto_final_gestion_de_datos_IA'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 77 (delta 23), reused 59 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 8.03 MiB | 10.64 MiB/s, done.
Resolving deltas: 100% (23/23), done.


In [3]:
# CREAR ESTRUCTURA DE CARPETAS
from pathlib import Path  # Librería para manejar rutas

# Carpeta base del proyecto
BASE = Path('/content/Proyecto_final_gestion_de_datos_IA/content')

# Lista de todas las carpetas que necesitamos
carpetas = [
    'datos/procesados',      # Datasets limpios (Guía 1 y 2)
    'datos/gobernanza',      # Dataset protegido (Guía 4)
    'datos/geo',             # Datasets geográficos (Guía 6)
    'documentos',            # Documentos generales
    'documentos/geo',        # Documentos geográficos
    'documentos/reporte',    # Informes (Guía 7)
    'evidencias',            # Capturas generales
    'evidencias/geo',        # Capturas geográficas
    'evidencias/reporte',    # Gráficos del informe
    'visualizaciones',       # Archivos de Power BI (Guía 5)
    'exportaciones',         # PDFs y DOCXs finales
    'templates',             # Plantillas Jinja2
    'scripts',               # Scripts del pipeline
]

# Crear cada carpeta
for c in carpetas:
    (BASE / c).mkdir(parents=True, exist_ok=True)

print("Estructura de carpetas creada:")
print(f"Base: {BASE}")
print("\nCarpetas disponibles:")
for c in carpetas:
    print(f"   ├──  {c}/")

Estructura de carpetas creada:
Base: /content/Proyecto_final_gestion_de_datos_IA/content

Carpetas disponibles:
   ├──  datos/procesados/
   ├──  datos/gobernanza/
   ├──  datos/geo/
   ├──  documentos/
   ├──  documentos/geo/
   ├──  documentos/reporte/
   ├──  evidencias/
   ├──  evidencias/geo/
   ├──  evidencias/reporte/
   ├──  visualizaciones/
   ├──  exportaciones/
   ├──  templates/
   ├──  scripts/


In [7]:
# IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt              # Gráficos estáticos (estable)
from pathlib import Path
from datetime import datetime
from jinja2 import Template
import base64
import json
import warnings
import os
warnings.filterwarnings('ignore')  # Ignorar warnings menores

# CONFIGURAR MATPLOTLIB
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print(" Librerías importadas")
print(f"   - pandas: {pd.__version__}")
print(f"   - matplotlib: {plt.matplotlib.__version__}")

# CARGAR DATASETS
# Procesados
df_limpio = pd.read_csv(BASE / 'datos' / 'procesados' / 'dataset_limpio.csv')
df_integrado = pd.read_csv(BASE / 'datos' / 'procesados' / 'dataset_integrado.csv')
df_tickets = pd.read_csv(BASE / 'datos' / 'procesados' / 'Support_tickets_processed.csv')
df_calidad = pd.read_excel(BASE / 'datos' / 'procesados' / 'reporte_calidad_ProyctoFinal02_equipo.xlsx')

# Geoespaciales
df_geo_general = pd.read_csv(BASE / 'datos' / 'geo' / 'dataset_devops_geografico.csv')
df_geo_downtime = pd.read_csv(BASE / 'datos' / 'geo' / 'dataset_devops_geografico_downtime.csv')
df_geo_empresa = pd.read_csv(BASE / 'datos' / 'geo' / 'dataset_devops_geografico_empresa.csv')
df_geo_resolution = pd.read_csv(BASE / 'datos' / 'geo' / 'dataset_devops_geografico_resolution.csv')
df_geo_sintetica = pd.read_csv(BASE / 'datos' / 'geo' / 'empresas_geografia_sintetica.csv')

# Gobernanza (Búsqueda automática por longitud de nombre)
carpeta_gobernanza = BASE / 'datos' / 'gobernanza'
archivo_gob = [f for f in os.listdir(carpeta_gobernanza) if f.startswith('dataset_publicable')][0]
df_gobernanza = pd.read_csv(carpeta_gobernanza / archivo_gob)

# Analíticos
df_analiticos = pd.read_parquet(BASE / 'datos' / 'analiticos' / 'devops_metrics_parquet_equipo')

# convertir columnas de fecha

df_limpio['event_date'] = pd.to_datetime(df_limpio['event_date'])
df_tickets['event_date'] = pd.to_datetime(df_tickets['event_date'])

# Validación segura para el dataset integrado
if 'event_date' in df_integrado.columns:
    df_integrado['event_date'] = pd.to_datetime(df_integrado['event_date'])

print(f"\nDatasets cargados:")
print(f"   - Procesado Limpio: {df_limpio.shape[0]} filas × {df_limpio.shape[1]} columnas")
print(f"   - Tickets Soporte: {df_tickets.shape[0]} filas × {df_tickets.shape[1]} columnas")
print(f"   - Calidad (Excel): {df_calidad.shape[0]} filas × {df_calidad.shape[1]} columnas")
print(f"   - Geo General: {df_geo_general.shape[0]} filas")
print(f"   - Geo Downtime: {df_geo_downtime.shape[0]} filas")
print(f"   - Geo Resolución: {df_geo_resolution.shape[0]} filas")
print(f"   - Gobernanza: {df_gobernanza.shape[0]} filas × {df_gobernanza.shape[1]} columnas")
print(f"   - Analíticos Parquet: {df_analiticos.shape[0]} filas × {df_analiticos.shape[1]} columnas")

print(f"\nPrimeras 3 filas del dataset principal (Limpio):")
print(df_limpio.head(3).to_string())

print(f"\nPrimeras 3 filas del dataset de Tickets:")
print(df_tickets.head(3).to_string())

 Librerías importadas
   - pandas: 2.2.3
   - matplotlib: 3.10.0

Datasets cargados:
   - Procesado Limpio: 20144 filas × 25 columnas
   - Tickets Soporte: 50000 filas × 34 columnas
   - Calidad (Excel): 25 filas × 4 columnas
   - Geo General: 20144 filas
   - Geo Downtime: 25 filas
   - Geo Resolución: 20144 filas
   - Gobernanza: 20144 filas × 11 columnas
   - Analíticos Parquet: 20144 filas × 25 columnas

Primeras 3 filas del dataset principal (Limpio):
   company_id product_area event_date  ticket_count  deployment_count  failed_deployments  avg_lead_time_hours  rollback_count  incident_count  avg_resolution_time_hours  total_downtime_min  avg_cpu_usage_pct  avg_memory_usage_pct  avg_response_time_ms  avg_error_rate_pct  avg_availability_pct  avg_requests_per_minute  deployment_failure_rate_pct  rollback_rate_pct  anio  mes  dia dia_semana  es_fin_semana  sin_datos_monitoreo
0      100001    Analytics 2026-01-01             3                 0                   0                  N

In [10]:
# CALCULAR KPIs (DevOps Metrics Analytics)
kpis = {
    'total_incidentes':       int(df_limpio['incident_count'].sum()),
    'total_tickets':          df_tickets['ticket_id'].nunique(),
    'downtime_total_horas':   df_limpio['total_downtime_min'].sum() / 60,
    'tiempo_resolucion_avg':  df_limpio['avg_resolution_time_hours'].mean(),
    'disponibilidad_avg':     df_limpio['avg_availability_pct'].mean(),
    'tasa_fallo_despliegues': df_limpio['deployment_failure_rate_pct'].mean(),
    'empresas_protegidas':    df_gobernanza['company_token'].nunique(),
    'departamentos_geo':      df_geo_downtime['departamento'].nunique(),
    'fecha_inicio':           df_limpio['event_date'].min().strftime('%d/%m/%Y'),
    'fecha_fin':              df_limpio['event_date'].max().strftime('%d/%m/%Y'),
    'periodo':                f"{df_limpio['event_date'].min().strftime('%d/%m/%Y')} - {df_limpio['event_date'].max().strftime('%d/%m/%Y')}"
}

print("KPIs del periodo devops")
print("=" * 50)
print(f"Total Incidentes:        {kpis['total_incidentes']:>10,}")
print(f"Tickets Procesados:      {kpis['total_tickets']:>10,}")
print(f"Downtime Total (Horas):  {kpis['downtime_total_horas']:>10,.1f} h")
print(f"Tiempo Resolución Prom.: {kpis['tiempo_resolucion_avg']:>10,.2f} h")
print(f"Disponibilidad Prom.:    {kpis['disponibilidad_avg']:>9.2f}%")
print(f"Tasa Fallo Despliegues:  {kpis['tasa_fallo_despliegues']:>9.2f}%")
print(f"Empresas Anonimizadas:   {kpis['empresas_protegidas']:>10,} (Guía 4)")
print(f"Deptos. Monitoreados:    {kpis['departamentos_geo']:>10,} (Guía 6)")
print(f"Periodo:                 {kpis['periodo']}")
print("=" * 50)

KPIs del periodo devops
Total Incidentes:             4,000
Tickets Procesados:          50,000
Downtime Total (Horas):    20,176.5 h
Tiempo Resolución Prom.:       6.31 h
Disponibilidad Prom.:        97.48%
Tasa Fallo Despliegues:       9.95%
Empresas Anonimizadas:           25 (Guía 4)
Deptos. Monitoreados:            14 (Guía 6)
Periodo:                 01/01/2026 - 30/06/2026


In [11]:
#  ANÁLISIS POR ÁREA DE PRODUCTO
df_producto = df_limpio.groupby('product_area').agg(
    total_incidentes=('incident_count', 'sum'),
    downtime_minutos=('total_downtime_min', 'sum'),
    tasa_fallo_avg=('deployment_failure_rate_pct', 'mean')
).reset_index()
df_producto = df_producto.sort_values('total_incidentes', ascending=False)

print(" ANÁLISIS POR ÁREA DE PRODUCTO:")
print(df_producto.to_string(index=False))

# 2. ANÁLISIS POR PRIORIDAD DE TICKET (Soporte)
df_prioridad = df_tickets.groupby('priority').agg(
    total_tickets=('ticket_id', 'count'),
    downtime_minutos=('downtime_min', 'sum'),
    usuarios_afectados=('customers_affected', 'sum')
).reset_index()
df_prioridad = df_prioridad.sort_values('total_tickets', ascending=False)

print("\n ANÁLISIS POR PRIORIDAD DE TICKET:")
print(df_prioridad.to_string(index=False))

# 3. ANÁLISIS GEOGRÁFICO POR DEPARTAMENTO (Guía 6)
# Usamos tu archivo específico de downtime geoespacial
df_departamento = df_geo_downtime.groupby('departamento').agg(
    downtime_total=('total_downtime_min', 'sum'),
    incidentes=('incident_count', 'sum'),
    fallos_despliegue=('failed_deployments', 'sum')
).reset_index()
df_departamento = df_departamento.sort_values('downtime_total', ascending=False)

print("\n ANÁLISIS GEOGRÁFICO POR DEPARTAMENTO (Downtime):")
print(df_departamento.head(10).to_string(index=False)) # Mostramos el top 10

# 4. ANÁLISIS DE GOBERNANZA (Guía 4)
# Verificamos los datos que fueron anonimizados (con tokens) por área
df_gob_resumen = df_gobernanza.groupby('product_area').agg(
    empresas_anonimizadas=('company_token', 'count'),
    tickets_protegidos=('ticket_count', 'sum'),
    incidentes_protegidos=('incident_count', 'sum')
).reset_index()

print("\n REPORTE DE GOBERNANZA (Datos Anonimizados):")
print(df_gob_resumen.to_string(index=False))

# 5. REPORTE DE CALIDAD DE DATOS (Excel)
# Mostramos un resumen de las columnas con problemas (si hay nulos)
df_calidad_resumen = df_calidad.sort_values('valores_nulos', ascending=False).head(5)

print("\n5. TOP 5 MÉTRICAS DE CALIDAD (Columnas con más nulos):")
print(df_calidad_resumen.to_string(index=False))

 ANÁLISIS POR ÁREA DE PRODUCTO:
 product_area  total_incidentes  downtime_minutos  tasa_fallo_avg
         Auth               790            231342       10.624690
       Mobile               776            237982       10.185736
    Analytics               761            237128        9.995837
Notifications               739            224268        9.326047
      Billing               563            169209       10.359524
Data Pipeline               371            110663        8.631248

 ANÁLISIS POR PRIORIDAD DE TICKET:
priority  total_tickets  downtime_minutos  usuarios_afectados
     low          25000            112138              603841
  medium          17500            241208             3342485
    high           7500            202436             5202669

 ANÁLISIS GEOGRÁFICO POR DEPARTAMENTO (Downtime):
departamento  downtime_total  incidentes  fallos_despliegue
 La Libertad          209140         689                251
San Salvador          174352         583           

In [15]:
# ===================================================================
# CONFIGURACIÓN INICIAL
# ===================================================================
ruta_reporte = BASE / 'evidencias' / 'reporte'
ruta_reporte.mkdir(parents=True, exist_ok=True)

# Preparar datos de tiempo
df_fecha = df_limpio.groupby('event_date')['total_downtime_min'].sum().reset_index()

# ===================================================================
# GRÁFICO 1: Tendencia temporal (líneas) - Downtime
# ===================================================================
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_fecha['event_date'], df_fecha['total_downtime_min'],
        marker='o', linewidth=2.5, markersize=10,
        color='#2c7bb6', markerfacecolor='#1a5490')
ax.fill_between(df_fecha['event_date'], df_fecha['total_downtime_min'], alpha=0.2, color='#2c7bb6')
ax.set_title('Evolución de Downtime Diario', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Fecha', fontsize=11)
ax.set_ylabel('Minutos de Inactividad', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')
ax.tick_params(axis='x', rotation=45)

# Añadir etiquetas con valores
for fecha, valor in zip(df_fecha['event_date'], df_fecha['total_downtime_min']):
    ax.annotate(f'{valor:,.0f}', (fecha, valor),
                textcoords="offset points", xytext=(0, 10),
                ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
ruta_downtime = ruta_reporte / 'grafico_downtime.png'
plt.savefig(ruta_downtime, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 1: {ruta_downtime.name}")

# ===================================================================
# GRÁFICO 2: Incidentes por Área (barras)
# ===================================================================
fig, ax = plt.subplots(figsize=(10, 6))
colores = plt.cm.Blues(np.linspace(0.4, 0.9, len(df_producto)))
barras = ax.bar(df_producto['product_area'], df_producto['total_incidentes'],
                color=colores, edgecolor='#1a5490', linewidth=1.5)
ax.set_title('Total de Incidentes por Área de Producto', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Área de Producto', fontsize=11)
ax.set_ylabel('Cantidad de Incidentes', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_producto['total_incidentes']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{valor:,.0f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_incidentes = ruta_reporte / 'grafico_incidentes.png'
plt.savefig(ruta_incidentes, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 2: {ruta_incidentes.name}")

# ===================================================================
# GRÁFICO 3: Tickets por Prioridad (barras)
# ===================================================================
fig, ax = plt.subplots(figsize=(10, 6))
colores_prioridad = ['#e74c3c', '#f1c40f', '#2ecc71'] # Rojo, Amarillo, Verde
barras = ax.bar(df_prioridad['priority'], df_prioridad['total_tickets'],
                color=colores_prioridad[:len(df_prioridad)], edgecolor='#2c3e50', linewidth=1.5)
ax.set_title('Distribución de Tickets por Prioridad', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Prioridad', fontsize=11)
ax.set_ylabel('Tickets', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_prioridad['total_tickets']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{valor:,.0f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_prioridad = ruta_reporte / 'grafico_prioridad.png'
plt.savefig(ruta_prioridad, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 3: {ruta_prioridad.name}")

# ===================================================================
# GRÁFICO 4: Top 5 Departamentos Críticos (barras)
# ===================================================================
top_deptos = df_departamento.head(5)
fig, ax = plt.subplots(figsize=(10, 6))
colores = plt.cm.Greens(np.linspace(0.4, 0.9, len(top_deptos)))
barras = ax.bar(top_deptos['departamento'], top_deptos['downtime_total'],
                color=colores, edgecolor='#1a5490', linewidth=1.5)
ax.set_title('Top 5 Departamentos con mayor Downtime', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Departamento', fontsize=11)
ax.set_ylabel('Minutos de Caída', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, top_deptos['downtime_total']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{valor:,.0f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_departamentos = ruta_reporte / 'grafico_departamentos.png'
plt.savefig(ruta_departamentos, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 4: {ruta_departamentos.name}")

# ===================================================================
# GRÁFICO 5: Gobernanza - Empresas Anonimizadas (barras)
# ===================================================================
fig, ax = plt.subplots(figsize=(10, 6))
colores = plt.cm.Purples(np.linspace(0.4, 0.9, len(df_gob_resumen)))
barras = ax.bar(df_gob_resumen['product_area'], df_gob_resumen['empresas_anonimizadas'],
                color=colores, edgecolor='#4a148c', linewidth=1.5)
ax.set_title('Gobernanza: Empresas Protegidas por Área', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Área de Producto', fontsize=11)
ax.set_ylabel('Empresas Anonimizadas', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_gob_resumen['empresas_anonimizadas']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{valor:,.0f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_gobernanza = ruta_reporte / 'grafico_gobernanza.png'
plt.savefig(ruta_gobernanza, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 5: {ruta_gobernanza.name}")

# ===================================================================
# GRÁFICO 6: Calidad de Datos - Valores Nulos (barras)
# ===================================================================
fig, ax = plt.subplots(figsize=(10, 6))
colores = plt.cm.Reds(np.linspace(0.4, 0.9, len(df_calidad_resumen)))
barras = ax.bar(df_calidad_resumen['columna'], df_calidad_resumen['valores_nulos'],
                color=colores, edgecolor='#b71c1c', linewidth=1.5)
ax.set_title('Calidad: Registros Nulos por Columna', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Nombre de Columna', fontsize=11)
ax.set_ylabel('Cantidad de Nulos', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_calidad_resumen['valores_nulos']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{valor:,.0f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_calidad = ruta_reporte / 'grafico_calidad.png'
plt.savefig(ruta_calidad, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 6: {ruta_calidad.name}")

print("\n¡Los 6 gráficos se generaron correctamente SIN reiniciar Colab!")

------ Gráfico 1: grafico_downtime.png
------ Gráfico 2: grafico_incidentes.png
------ Gráfico 3: grafico_prioridad.png
------ Gráfico 4: grafico_departamentos.png
------ Gráfico 5: grafico_gobernanza.png
------ Gráfico 6: grafico_calidad.png

¡Los 6 gráficos se generaron correctamente SIN reiniciar Colab!


In [16]:
# VERIFICAR LOS 6 GRÁFICOS
graficos = {
    'Downtime':      ruta_downtime,
    'Incidentes':    ruta_incidentes,
    'Prioridad':     ruta_prioridad,
    'Departamentos': ruta_departamentos,
    'Gobernanza':    ruta_gobernanza,
    'Calidad':       ruta_calidad,
}

print("VERIFICANDO GRÁFICOS:\n")
for nombre, ruta in graficos.items():
    if ruta.exists():
        tamaño = ruta.stat().st_size / 1024
        print(f"OK {nombre}: {ruta.name} ({tamaño:.1f} KB)")
    else:
        print(f"FAILED {nombre}: NO ENCONTRADO")

# DESCARGAR
from google.colab import files
print("\nDescargando gráficos...")
for nombre, ruta in graficos.items():
    if ruta.exists():
        files.download(str(ruta))

VERIFICANDO GRÁFICOS:

OK Downtime: grafico_downtime.png (298.6 KB)
OK Incidentes: grafico_incidentes.png (64.0 KB)
OK Prioridad: grafico_prioridad.png (39.0 KB)
OK Departamentos: grafico_departamentos.png (67.9 KB)
OK Gobernanza: grafico_gobernanza.png (70.2 KB)
OK Calidad: grafico_calidad.png (64.2 KB)

Descargando gráficos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
# INSTALAR HARFBUZZ-SUBSET
!apt-get update -qq
!apt-get install -y libharfbuzz-subset0 -qq

# Reinstalar WeasyPrint con soporte para HarfBuzz
!pip install -U weasyprint -q

print("HarfBuzz-Subset instalado")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libharfbuzz-subset0:amd64.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../libharfbuzz-subset0_8.3.0-2build2_amd64.deb ...
Unpacking libharfbuzz-subset0:amd64 (8.3.0-2build2) ...
Setting up libharfbuzz-subset0:amd64 (8.3.0-2build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb

In [19]:
# TEMPLATE HTML CON JINJA2 (Adaptado a DevOps Metrics Analytics)
from jinja2 import Template
import datetime

template_informe = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Informe Gerencial - DevOps Metrics Analytics</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 40px; color: #333; line-height: 1.6; }
        h1 { color: #1a5490; border-bottom: 3px solid #1a5490; padding-bottom: 10px; }
        h2 { color: #2c7bb6; margin-top: 30px; border-bottom: 1px solid #ddd; padding-bottom: 5px; }
        h3 { color: #444; margin-top: 20px; }
        .portada { text-align: center; margin-bottom: 40px; padding: 25px; background: #f0f7ff; border-radius: 8px; }
        .portada h1 { border: none; font-size: 28px; margin-bottom: 5px; }
        .portada h2 { border: none; color: #666; margin-top: 5px; }
        .kpi-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px; margin: 20px 0; }
        .kpi-card { background: #f0f7ff; padding: 15px; border-left: 4px solid #2c7bb6; border-radius: 4px; }
        .kpi-valor { font-size: 24px; font-weight: bold; color: #1a5490; }
        .kpi-label { font-size: 12px; color: #666; text-transform: uppercase; margin-bottom: 5px; }
        .hallazgo { background: #f9f9f9; padding: 15px; border-left: 4px solid #ff9800; margin: 15px 0; border-radius: 4px; }
        .hallazgo h3 { margin-top: 0; color: #e65100; }
        .limitacion { background: #fff3cd; padding: 10px; border-left: 4px solid #ffc107; margin: 10px 0; border-radius: 4px; }
        .limitacion-titulo { font-weight: bold; color: #856404; margin-right: 5px; }
        table { border-collapse: collapse; width: 100%; margin: 15px 0; }
        th, td { border: 1px solid #ddd; padding: 10px; text-align: left; }
        th { background: #2c7bb6; color: white; }
        tr:nth-child(even) { background: #f9f9f9; }
        .pie { font-size: 11px; color: #999; margin-top: 40px; border-top: 1px solid #ddd; padding-top: 10px; text-align: center; }
        img { max-width: 100%; margin: 10px 0; border: 1px solid #eee; border-radius: 4px; }
        .metodologia { background: #e8f5e9; padding: 15px; border-radius: 4px; margin: 15px 0; }
    </style>
</head>
<body>

<div class="portada">
    <h1>INFORME GERENCIAL OPERATIVO</h1>
    <h2>DevOps Metrics Analytics</h2>
    <p><strong>Periodo:</strong> {{ kpis.periodo }}</p>
    <p><strong>Fecha de generación:</strong> {{ fecha_generacion }}</p>
    <p><strong>Versión:</strong> 1.0 (Final)</p>
    <p><strong>Desarrollador:</strong> Alan Aguirre</p>
</div>

<h2>1. Resumen Ejecutivo</h2>
<p>El presente informe analiza el desempeño de la infraestructura tecnológica y soporte durante el periodo {{ kpis.periodo }}.
Su propósito es apoyar la toma de decisiones gerenciales sobre asignación de recursos, mejora continua de software e identificar puntos críticos a nivel nacional,
a partir de la integración de registros transaccionales, auditorías de calidad (Guía 2), políticas de gobernanza (Guía 4) y análisis geoespacial (Guía 6).</p>

<h3>Indicadores Principales</h3>
<div class="kpi-grid">
    <div class="kpi-card">
        <div class="kpi-label">Total Incidentes</div>
        <div class="kpi-valor">{{ "{:,}".format(kpis.total_incidentes) }}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Downtime Total (Horas)</div>
        <div class="kpi-valor">{{ "{:,.1f}".format(kpis.downtime_total_horas) }} h</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Tiempo Resolución Prom.</div>
        <div class="kpi-valor">{{ "{:,.2f}".format(kpis.tiempo_resolucion_avg) }} h</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Disponibilidad Prom.</div>
        <div class="kpi-valor">{{ "%.2f"|format(kpis.disponibilidad_avg) }}%</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Tickets Procesados</div>
        <div class="kpi-valor">{{ "{:,}".format(kpis.total_tickets) }}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Empresas Protegidas (Gob.)</div>
        <div class="kpi-valor">{{ "{:,}".format(kpis.empresas_protegidas) }}</div>
    </div>
</div>

<h2>2. Contexto y Metodología</h2>
<div class="metodologia">
    <p><strong>Fuentes utilizadas:</strong> Datasets procesados Parquet (Guía 2), Políticas de Anonimización (Guía 4), Reporte de Calidad Excel, Análisis Geoespacial (Guía 6).</p>
    <p><strong>Proceso:</strong> Extracción → Limpieza → Enmascaramiento de Datos (Gobernanza) → Análisis Territorial → Visualización Gerencial.</p>
    <p><strong>Filtros aplicados:</strong> Periodo {{ kpis.periodo }}, todos los departamentos, áreas de producto y prioridades.</p>
</div>

<h2>3. Resultados Generales</h2>

<h3>3.1 Evolución de Inactividad del Sistema</h3>
<img src="{{ ruta_downtime }}" alt="Tendencia de downtime">
<p><strong>Lectura:</strong> Se presenta el volumen de minutos de inactividad (downtime) diario, identificando picos críticos que requirieron intervención inmediata.</p>

<h3>3.2 Incidentes por Área de Producto</h3>
<img src="{{ ruta_incidentes }}" alt="Incidentes por área">
<p><strong>Lectura:</strong> El módulo "Auth" genera los mayores incidentes y el mayor tiempo de caída, requiriendo refactorización urgente.</p>

<h3>3.3 Tickets de Soporte por Prioridad</h3>
<img src="{{ ruta_prioridad }}" alt="Prioridad de tickets">
<p><strong>Lectura:</strong> La mayor carga de tickets se concentra en nivel bajo, pero los de prioridad Alta afectan a más de 5 millones de usuarios.</p>

<h2>4. Análisis Geoespacial e Infraestructura</h2>
<img src="{{ ruta_departamentos }}" alt="Downtime por departamento">
<p><strong>Patrón observado:</strong> El departamento de La Libertad concentra el mayor tiempo de caídas y fallos de despliegue a nivel nacional, superando a San Salvador.</p>
<p><strong>Lectura responsable:</strong> Existe una concentración de actividad técnica en zonas metropolitanas. Los datos geográficos determinan los nodos críticos pero no demuestran causalidad directa sobre la arquitectura de la red.</p>

<h2>5. Auditoría y Gobernanza</h2>
<img src="{{ ruta_gobernanza }}" alt="Gobernanza">
<p><strong>Cumplimiento:</strong> Se aplicó enmascaramiento mediante `company_token` a más de 18,000 registros transaccionales, cumpliendo la Ley de Protección de Datos (Guía 4).</p>

<img src="{{ ruta_calidad }}" alt="Calidad de datos">
<p><strong>Auditoría:</strong> Se detectaron valores nulos en columnas secundarias que no afectaron los cálculos de KPIs principales, documentados en el Excel de calidad.</p>

<h2>6. Hallazgos Principales</h2>

<div class="hallazgo">
    <h3>1. Cuello de botella en Autenticación</h3>
    <p><strong>Evidencia:</strong> El área "Auth" presenta 790 incidentes y más de 230 mil minutos de downtime.</p>
    <p><strong>Interpretación:</strong> La capa de seguridad está fallando o saturándose, afectando el acceso general al sistema.</p>
    <p><strong>Impacto:</strong> Afectación directa a la disponibilidad percibida por el usuario final.</p>
    <p><strong>Decisión sugerida:</strong> Auditar los servidores de autenticación y escalar verticalmente los recursos de ese nodo.</p>
    <p><em>Limitación: No se tienen logs detallados del tipo de error de autenticación (timeout vs. password incorrecto).</em></p>
</div>

<div class="hallazgo">
    <h3>2. Riesgo Crítico en La Libertad</h3>
    <p><strong>Evidencia:</strong> La Libertad registró 209,140 minutos de downtime, siendo el departamento más afectado.</p>
    <p><strong>Interpretación:</strong> La infraestructura de conectividad o el centro de datos local está inestable.</p>
    <p><strong>Impacto:</strong> Disrupción del servicio en un área económica clave del país.</p>
    <p><strong>Decisión sugerida:</strong> Desplegar equipo técnico a La Libertad para revisar infraestructura física y routing.</p>
    <p><em>Limitación: No se especifica si las caídas fueron por hardware propio o proveedor de internet de terceros.</em></p>
</div>

<h2>7. Recomendaciones</h2>
<table>
    <thead>
        <tr>
            <th>#</th>
            <th>Acción</th>
            <th>Prioridad</th>
            <th>Responsable</th>
            <th>Plazo</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>1</td>
            <td>Refactorizar módulo Auth</td>
            <td>Alta</td>
            <td>Ingeniería Backend</td>
            <td>1 Mes</td>
        </tr>
        <tr>
            <td>2</td>
            <td>Revisar nodos en La Libertad</td>
            <td>Media-Alta</td>
            <td>Infraestructura de Redes</td>
            <td>2 Semanas</td>
        </tr>
        <tr>
            <td>3</td>
            <td>Mejorar tiempos de resolución (SLA)</td>
            <td>Media</td>
            <td>Soporte Técnico Nivel 2</td>
            <td>Continua</td>
        </tr>
    </tbody>
</table>

<h2>8. Limitaciones del Análisis</h2>
<div class="limitacion"><span class="limitacion-titulo">[!]</span> Las métricas de disponibilidad asumen que todos los módulos son críticos.</div>
<div class="limitacion"><span class="limitacion-titulo">[!]</span> Los datos geográficos están mapeados al cliente afectado, no a la ubicación del servidor que falló.</div>

<h2>9. Conclusiones</h2>
<p>El análisis operativo del periodo {{ kpis.periodo }} muestra una infraestructura bajo estrés en sus componentes críticos ("Auth" y "Mobile"), manteniendo un tiempo de resolución de {{ "{:,.1f}".format(kpis.tiempo_resolucion_avg) }} horas en promedio.
Las políticas de gobernanza fueron un éxito, asegurando el anonimato de los clientes comerciales.
Geográficamente, es indispensable focalizar los esfuerzos de estabilidad en La Libertad y San Salvador para mejorar la experiencia nacional.</p>

<div class="pie">
    <p>Informe generado automáticamente por el pipeline de reportería gerencial - DevOps Metrics Analytics</p>
    <p>Proyecto Integrador COIDS1604 - Desarrollo de Software</p>
</div>

</body>
</html>
"""

print("Template Jinja2 definido (adaptado a DevOps Metrics )")
print(f"Longitud: {len(template_informe):,} caracteres")

Template Jinja2 definido (adaptado a DevOps Metrics )
Longitud: 9,872 caracteres


In [20]:
import base64
from datetime import datetime
from jinja2 import Template

# 1. FUNCIÓN PARA CONVERTIR IMAGEN A BASE64 (Obligatorio para que WeasyPrint las vea)
def imagen_a_base64(ruta):
    """Convierte una imagen a base64 para incrustarla directamente en HTML."""
    with open(ruta, 'rb') as f:
        return f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"

# 2. PREPARAR DATOS PARA EL TEMPLATE (Métricas DevOps)
datos_template = {
    'periodo': kpis['periodo'],
    'fecha_generacion': datetime.now().strftime('%d/%m/%Y %H:%M'),
    'version': '1.0 (Final)',
    'equipo': 'Equipo DevOps Metrics Analytics',
    'kpis': type('KPIs', (), kpis)(),

    # Imágenes en base64 (Los 6 gráficos)
    'ruta_downtime': imagen_a_base64(ruta_downtime),
    'ruta_incidentes': imagen_a_base64(ruta_incidentes),
    'ruta_prioridad': imagen_a_base64(ruta_prioridad),
    'ruta_departamentos': imagen_a_base64(ruta_departamentos),
    'ruta_gobernanza': imagen_a_base64(ruta_gobernanza),
    'ruta_calidad': imagen_a_base64(ruta_calidad),

    # Hallazgos dinámicos basados en los datos reales
    'hallazgos': [
        {
            'titulo': f'El módulo {df_producto.iloc[0]["product_area"]} concentra el mayor número de incidentes',
            'evidencia': f'El área {df_producto.iloc[0]["product_area"]} generó {df_producto.iloc[0]["total_incidentes"]} incidentes, acumulando {df_producto.iloc[0]["downtime_minutos"]:,.0f} minutos de caída.',
            'interpretacion': 'El servicio de autenticación y acceso es inestable bajo la carga actual.',
            'impacto': 'Bloquea el uso de todas las demás herramientas de la plataforma.',
            'decision': 'Escalar recursos y refactorizar código de Auth de inmediato.',
            'limitacion': 'No se cuenta con logs de red detallados para saber si fue un ataque DDoS.'
        },
        {
            'titulo': f'El departamento de {top_deptos.iloc[0]["departamento"]} es el más inestable geográficamente',
            'evidencia': f'Se registraron {top_deptos.iloc[0]["downtime_total"]:,.0f} minutos de downtime en {top_deptos.iloc[0]["departamento"]}.',
            'interpretacion': 'Problemas serios de enrutamiento o fallos en el centro de datos de esa zona.',
            'impacto': 'Degrada la experiencia de usuario a nivel regional.',
            'decision': 'Desplegar equipo técnico presencial a nodos de conectividad.',
            'limitacion': 'El análisis asume que la IP reportada corresponde al lugar real del fallo físico.'
        },
        {
            'titulo': 'Alta carga de tickets de nivel bajo',
            'evidencia': f'Existen {df_prioridad.iloc[0]["total_tickets"]:,.0f} tickets procesados con prioridad "{df_prioridad.iloc[0]["priority"]}".',
            'interpretacion': 'El equipo de soporte invierte mucho tiempo en tareas menores.',
            'impacto': 'Posible descuido de los {df_prioridad.iloc[2]["total_tickets"]:,.0f} tickets de prioridad Alta que afectan a 5 millones de usuarios.',
            'decision': 'Implementar un chatbot de IA para resolver automáticamente tickets Low.',
            'limitacion': 'No se analizó el texto de los tickets para confirmar que son automatizables.'
        }
    ],

    # Recomendaciones gerenciales
    'recomendaciones': [
        {'accion': f'Refactorizar el microservicio de {df_producto.iloc[0]["product_area"]}', 'prioridad': 'Crítica', 'responsable': 'DevOps / Backend', 'plazo': 'Inmediato'},
        {'accion': f'Revisar infraestructura física en {top_deptos.iloc[0]["departamento"]}', 'prioridad': 'Alta', 'responsable': 'Ing. Redes', 'plazo': '1 Semana'},
        {'accion': 'Automatizar tickets de baja prioridad (Nivel 1)', 'prioridad': 'Media', 'responsable': 'Soporte IT', 'plazo': '3 Meses'},
        {'accion': 'Mantener políticas de anonimización (Company Token)', 'prioridad': 'Alta', 'responsable': 'Gobernanza de Datos', 'plazo': 'Continuo'},
        {'accion': 'Corregir valores nulos reportados en auditoría de calidad', 'prioridad': 'Baja', 'responsable': 'Data Engineering', 'plazo': '1 Mes'}
    ],

    # Limitaciones del análisis
    'limitaciones': [
        'Correlación espacial no implica causalidad física (Guía 6).',
        'Las métricas de disponibilidad asumen un peso igual para todos los servicios.',
        'La calidad de los datos depende de los reportes manuales de soporte.',
        'El análisis de gobernanza solo cubre PII a nivel empresarial, no correos individuales.',
        'Los costos operativos de mantenimiento no están incluidos en el modelo.'
    ]
}

# 3. RENDERIZAR TEMPLATE
template = Template(template_informe)
html_renderizado = template.render(**datos_template)

# Asegurar que exista la carpeta de exportaciones
ruta_exportaciones = BASE / 'exportaciones'
ruta_exportaciones.mkdir(parents=True, exist_ok=True)

# Guardar HTML
ruta_html = ruta_exportaciones / 'Reporte_DevOps_Metrics.html'
ruta_html.write_text(html_renderizado, encoding='utf-8')
print(f"HTML generado: {ruta_html.name}")

# 4. CONVERTIR A PDF CON WEASYPRINT
try:
    from weasyprint import HTML
    ruta_pdf = ruta_exportaciones / 'Reporte_Gerencial_DevOps_v1.0.pdf'
    HTML(string=html_renderizado).write_pdf(str(ruta_pdf))
    print(f"PDF generado exitosamente: {ruta_pdf.name}")
    print(f"Tamaño: {ruta_pdf.stat().st_size / 1024:.1f} KB")

    # 5. DESCARGAR AUTOMÁTICAMENTE
    from google.colab import files
    files.download(str(ruta_pdf))
    print("Iniciando descarga del PDF...")
except Exception as e:
    print(f"Error al generar el PDF: {e}")
    print("Sin embargo, el archivo HTML se generó correctamente. Puedes abrirlo y guardarlo como PDF en tu navegador.")

HTML generado: Reporte_DevOps_Metrics.html
PDF generado exitosamente: Reporte_Gerencial_DevOps_v1.0.pdf
Tamaño: 586.9 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Iniciando descarga del PDF...


In [21]:
# GENERAR DOCX CON PYTHON-DOCX
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime

doc = Document()

# Título
titulo = doc.add_heading('INFORME GERENCIAL OPERATIVO', 0)
titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

subtitulo = doc.add_heading('DevOps Metrics Analytics', level=1)
subtitulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_paragraph(f'Periodo: {kpis["periodo"]}')
doc.add_paragraph(f'Fecha de generación: {datetime.now().strftime("%d/%m/%Y %H:%M")}')
doc.add_paragraph('Versión: 1.0 (Final)')
doc.add_paragraph('Desarrollador: Alan Aguirre')

# Resumen ejecutivo
doc.add_heading('1. Resumen Ejecutivo', level=2)
doc.add_paragraph(f'El presente informe analiza el desempeño de la infraestructura tecnológica y soporte durante el periodo {kpis["periodo"]}.')

# KPIs
doc.add_heading('Indicadores Principales', level=3)
tabla_kpis = doc.add_table(rows=6, cols=2)
tabla_kpis.style = 'Light Grid Accent 1'

datos_kpis = [
    ('Total Incidentes', f'{kpis["total_incidentes"]:,}'),
    ('Downtime Total (Horas)', f'{kpis["downtime_total_horas"]:,.1f} h'),
    ('Tiempo Resolución Prom.', f'{kpis["tiempo_resolucion_avg"]:,.2f} h'),
    ('Disponibilidad Prom.', f'{kpis["disponibilidad_avg"]:.2f}%'),
    ('Tickets Procesados', f'{kpis["total_tickets"]:,}'),
    ('Empresas Anonimizadas', f'{kpis["empresas_protegidas"]:,}')
]

for i, (label, valor) in enumerate(datos_kpis):
    tabla_kpis.rows[i].cells[0].text = label
    tabla_kpis.rows[i].cells[1].text = valor

# Resultados con gráficos
doc.add_heading('2. Resultados Generales', level=2)

doc.add_heading('2.1 Evolución de Inactividad del Sistema', level=3)
doc.add_picture(str(ruta_downtime), width=Inches(6))

doc.add_heading('2.2 Incidentes por Área de Producto', level=3)
doc.add_picture(str(ruta_incidentes), width=Inches(6))

doc.add_heading('2.3 Tickets de Soporte por Prioridad', level=3)
doc.add_picture(str(ruta_prioridad), width=Inches(6))

doc.add_heading('2.4 Top 5 Departamentos Críticos', level=3)
doc.add_picture(str(ruta_departamentos), width=Inches(6))

doc.add_heading('2.5 Auditoría y Gobernanza', level=3)
doc.add_picture(str(ruta_gobernanza), width=Inches(6))
doc.add_picture(str(ruta_calidad), width=Inches(6))

# Hallazgos
doc.add_heading('3. Hallazgos Principales', level=2)
for i, h in enumerate(datos_template['hallazgos'], 1):
    doc.add_heading(f'{i}. {h["titulo"]}', level=3)
    doc.add_paragraph(f'Evidencia: {h["evidencia"]}')
    doc.add_paragraph(f'Interpretación: {h["interpretacion"]}')
    doc.add_paragraph(f'Impacto: {h["impacto"]}')
    doc.add_paragraph(f'Decisión: {h["decision"]}')

# Recomendaciones
doc.add_heading('4. Recomendaciones', level=2)
tabla_rec = doc.add_table(rows=1, cols=5)
tabla_rec.style = 'Light Grid Accent 1'
headers = ['#', 'Acción', 'Prioridad', 'Responsable', 'Plazo']
for i, h in enumerate(headers):
    tabla_rec.rows[0].cells[i].text = h

for i, r in enumerate(datos_template['recomendaciones'], 1):
    row = tabla_rec.add_row()
    row.cells[0].text = str(i)
    row.cells[1].text = r['accion']
    row.cells[2].text = r['prioridad']
    row.cells[3].text = r['responsable']
    row.cells[4].text = r['plazo']

# Limitaciones
doc.add_heading('5. Limitaciones', level=2)
for l in datos_template['limitaciones']:
    doc.add_paragraph(l, style='List Bullet')

# Guardar
ruta_docx = BASE / 'exportaciones' / 'Reporte_Gerencial_DevOps_v1.0.docx'
doc.save(str(ruta_docx))
print(f"DOCX generado: {ruta_docx.name}")
print(f"Tamaño: {ruta_docx.stat().st_size / 1024:.1f} KB")

from google.colab import files
files.download(str(ruta_docx))
print("DOCX descargado exitosamente.")

DOCX generado: Reporte_Gerencial_DevOps_v1.0.docx
Tamaño: 567.1 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DOCX descargado exitosamente.


In [22]:
import json
from datetime import datetime

# METADATA DE TRAZABILIDAD (DevOps Metrics Analytics)
metadata = {
    'version': '1.0',
    'fecha_generacion': datetime.now().isoformat(),
    'periodo_analizado': kpis['periodo'],
    'fuentes_datos': [
        'dataset_limpio.csv (Guía 2 - Transaccional)',
        'Support_tickets_processed.csv (Guía 2 - Soporte)',
        'Dataset anonimizado con tokens (Guía 4 - Gobernanza)',
        'dataset_devops_geografico_downtime.csv (Guía 6 - Geoespacial)',
        'reporte_calidad_ProyctoFinal02_equipo.xlsx (Auditoría Calidad)'
    ],
    'filtros_aplicados': {
        'periodo': kpis['periodo'],
        'areas_producto': 'todas',
        'prioridades_ticket': 'todas',
        'departamentos_geo': 'todos'
    },
    'kpis': {
        'total_incidentes': int(kpis['total_incidentes']),
        'downtime_total_horas': float(kpis['downtime_total_horas']),
        'tiempo_resolucion_avg': float(kpis['tiempo_resolucion_avg']),
        'disponibilidad_avg': float(kpis['disponibilidad_avg']),
        'empresas_anonimizadas': int(kpis['empresas_protegidas'])
    },
    'archivos_generados': [
        'Reporte_Gerencial_DevOps_v1.0.pdf',
        'Reporte_Gerencial_DevOps_v1.0.docx',
        'Reporte_DevOps_Metrics.html'
    ],
    'generado_por': 'pipeline_reporteria_devops.py'
}

# Guardar el archivo JSON
ruta_metadata = BASE / 'exportaciones' / 'metadata_informe.json'
ruta_metadata.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

print("Metadata de trazabilidad guardada correctamente en:")
print(f"Ruta: {ruta_metadata}\n")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

# Descargar la metadata
from google.colab import files
files.download(str(ruta_metadata))

Metadata de trazabilidad guardada correctamente en:
Ruta: /content/Proyecto_final_gestion_de_datos_IA/content/exportaciones/metadata_informe.json

{
  "version": "1.0",
  "fecha_generacion": "2026-10-08T05:18:48.637770",
  "periodo_analizado": "01/01/2026 - 30/06/2026",
  "fuentes_datos": [
    "dataset_limpio.csv (Guía 2 - Transaccional)",
    "Support_tickets_processed.csv (Guía 2 - Soporte)",
    "Dataset anonimizado con tokens (Guía 4 - Gobernanza)",
    "dataset_devops_geografico_downtime.csv (Guía 6 - Geoespacial)",
    "reporte_calidad_ProyctoFinal02_equipo.xlsx (Auditoría Calidad)"
  ],
  "filtros_aplicados": {
    "periodo": "01/01/2026 - 30/06/2026",
    "areas_producto": "todas",
    "prioridades_ticket": "todas",
    "departamentos_geo": "todos"
  },
  "kpis": {
    "total_incidentes": 4000,
    "downtime_total_horas": 20176.533333333333,
    "tiempo_resolucion_avg": 6.314362974161626,
    "disponibilidad_avg": 97.48086272026532,
    "empresas_anonimizadas": 25
  },
  "arch

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# CREAR GITHUB ACTIONS WORKFLOW (DevOps Metrics Analytics)
workflow = """
name: Generar Reporte DevOps Automático

on:
  schedule:
    - cron: '0 8 * * 1'  # Todos los lunes a las 8 AM
  workflow_dispatch:     # Permite ejecución manual

jobs:
  generar-reporte:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      - name: Instalar dependencias del sistema y Python
        run: |
          sudo apt-get update
          sudo apt-get install -y libharfbuzz-subset0
          pip install pandas matplotlib jinja2 weasyprint python-docx pyarrow openpyxl
      - name: Ejecutar pipeline de reporteria DevOps
        run: python scripts/pipeline_reporteria_devops.py
      - name: Subir artefactos
        uses: actions/upload-artifact@v3
        with:
          name: reportes-devops-generados
          path: exportaciones/
"""

ruta_workflow = BASE / '.github' / 'workflows'
ruta_workflow.mkdir(parents=True, exist_ok=True)
archivo_yml = ruta_workflow / 'generar_reporte_devops.yml'
archivo_yml.write_text(workflow, encoding='utf-8')

print("Workflow de GitHub Actions creado exitosamente.")
print(f"Ruta: {archivo_yml}")

# Opcional: Descargar el archivo YAML para tu repositorio local
from google.colab import files
files.download(str(archivo_yml))

Workflow de GitHub Actions creado exitosamente.
Ruta: /content/Proyecto_final_gestion_de_datos_IA/content/.github/workflows/generar_reporte_devops.yml


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>